# Assignment 6: Build and Evaluate Tree Models

Juan Maldonado Franco  
DDS-8555 Predictive Analysis  
Mohamed Nabeel

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "DDS-8555 - Predictive Analysis").exists():
        COURSE = parent / "DDS-8555 - Predictive Analysis"
        break
else:
    COURSE = ROOT.parents[1]
DATA = COURSE / "data"
KAGGLE = DATA / "kaggle"
SUBMISSIONS = DATA / "submissions"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 80)

## Conceptual Question 1

A six-region recursive binary split can be created by first splitting X1 at t1, then splitting the left side on X2 at t2, and continuing with additional splits inside selected regions.  The matching decision tree starts with the first X1 split at the root, then branches into the later X2 and X1 cuts.  The important point is that each rectangular region corresponds to one terminal node, and each internal node corresponds to one binary decision.

## Applied Question 12 and Kaggle Tree Models

The applied exercise asks for boosting, bagging, random forests, and BART on a chosen data set.  The Kaggle Obesity data is used here because it also satisfies the assignment competition requirement for decision tree, bagged tree, random forest, and boosted tree submissions.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

obesity = pd.read_csv(KAGGLE / "playground-series-s4e2" / "train.csv")
X = obesity.drop(columns=["NObeyesdad"])
y = obesity["NObeyesdad"]
display(y.value_counts(normalize=True).rename("class_share").to_frame())
cat = X.select_dtypes(include="object").columns.tolist()
num = [c for c in X.columns if c not in cat + ["id"]]
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat)], remainder="passthrough")
X_train, X_valid, y_train, y_valid = train_test_split(X.drop(columns=["id"]), y, test_size=.2, stratify=y, random_state=RANDOM_STATE)
base_tree = DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)
models = {
    "Decision tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    "Bagging": BaggingClassifier(estimator=base_tree, n_estimators=80, random_state=RANDOM_STATE, n_jobs=-1),
    "Random forest": RandomForestClassifier(n_estimators=150, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
rows = []
fitted_models = {}
for name, clf in models.items():
    model = Pipeline([("pre", pre), ("model", clf)])
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    pred = model.predict(X_valid)
    fitted_models[name] = (model, pred)
    rows.append({
        "model": name,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "validation_accuracy": accuracy_score(y_valid, pred),
        "balanced_accuracy": balanced_accuracy_score(y_valid, pred),
        "generalization_gap": accuracy_score(y_train, train_pred) - accuracy_score(y_valid, pred),
    })
validation = pd.DataFrame(rows).sort_values("validation_accuracy", ascending=False)
display(validation)

best_name = validation.iloc[0]["model"]
best_model, best_pred = fitted_models[best_name]
display(pd.DataFrame(confusion_matrix(y_valid, best_pred, labels=best_model.classes_), index=best_model.classes_, columns=best_model.classes_))

feature_names = best_model.named_steps["pre"].get_feature_names_out()
if hasattr(best_model.named_steps["model"], "feature_importances_"):
    importances = pd.DataFrame({
        "feature": feature_names,
        "importance": best_model.named_steps["model"].feature_importances_,
    }).sort_values("importance", ascending=False)
    display(importances.head(12))

,class_share
NObeyesdad,
Obesity_Type_III,0.194913
Obesity_Type_II,0.156470
Normal_Weight,0.148473
Obesity_Type_I,0.140187
Insufficient_Weight,0.121544
Overweight_Level_II,0.121495
Overweight_Level_I,0.116919


,model,train_accuracy,validation_accuracy,balanced_accuracy,generalization_gap
3,Gradient boosting,0.921233,0.905347,0.894882,0.015886
2,Random forest,0.952668,0.890173,0.877267,0.062494
1,Bagging,0.902565,0.888006,0.875682,0.014560
0,Decision tree,0.893171,0.869461,0.856238,0.023711


,Insufficient_Weight,Normal_Weight,Obesity_Type_I,Obesity_Type_II,Obesity_Type_III,Overweight_Level_I,Overweight_Level_II
Insufficient_Weight,482,22,0,0,0,1,0
Normal_Weight,32,547,1,0,0,29,8
Obesity_Type_I,1,3,523,14,3,7,31
Obesity_Type_II,1,0,20,628,0,0,1
Obesity_Type_III,0,0,2,1,806,0,0
Overweight_Level_I,1,48,9,0,0,365,62
Overweight_Level_II,0,7,41,3,0,45,408


,feature,importance
24,remainder__Weight,0.608735
25,remainder__FCVC,0.111034
23,remainder__Height,0.072169
1,cat__Gender_Male,0.049671
0,cat__Gender_Female,0.048556
22,remainder__Age,0.036326
27,remainder__CH2O,0.016228
16,cat__CALC_no,0.013646
26,remainder__NCP,0.011562
29,remainder__TUE,0.007041


In [3]:
import re

status_path = SUBMISSIONS / "kaggle_submission_status_playground-series-s4e2.txt"
for encoding in ("utf-16", "utf-8"):
    try:
        text = status_path.read_text(encoding=encoding)
        break
    except UnicodeError:
        continue

records = []
for line in text.splitlines():
    parts = re.split(r"\s{2,}", line.strip())
    if len(parts) >= 7 and parts[0].isdigit() and "A6_" in parts[1]:
        records.append({
            "ref": parts[0],
            "fileName": parts[1],
            "date": parts[2],
            "description": parts[3],
            "status": parts[4],
            "publicScore": parts[5],
            "privateScore": parts[6],
        })

display(pd.DataFrame(records))
display(pd.DataFrame({
    "submission_file": ['A6_decision_tree_playground_series_s4e2.csv', 'A6_bagging_playground_series_s4e2.csv', 'A6_random_forest_playground_series_s4e2.csv', 'A6_gradient_boosting_playground_series_s4e2.csv'],
    "exists_locally": [(SUBMISSIONS / file).exists() for file in ['A6_decision_tree_playground_series_s4e2.csv', 'A6_bagging_playground_series_s4e2.csv', 'A6_random_forest_playground_series_s4e2.csv', 'A6_gradient_boosting_playground_series_s4e2.csv']],
}))
display(pd.DataFrame({
    "evidence": ["Public GitHub repository", "Notebook path in repository"],
    "value": [
        "https://github.com/maldo81/dds-8555-predictive-analysis",
        "Week 6/Assignment 6/MaldonadoJDDS8555-6.ipynb",
    ],
}))

,ref,fileName,date,description,status,publicScore,privateScore
0,52966810,A6_gradient_boosting_playground_series_s4e2.csv,2026-05-23 21:27:41.547000,DDS-8555 A6 boosted tree model,SubmissionStatus.COMPLETE,0.90462,0.90245
1,52966809,A6_random_forest_playground_series_s4e2.csv,2026-05-23 21:27:39.150000,DDS-8555 A6 random forest model,SubmissionStatus.COMPLETE,0.88945,0.89315
2,52966808,A6_bagging_playground_series_s4e2.csv,2026-05-23 21:27:36.670000,DDS-8555 A6 bagged tree model,SubmissionStatus.COMPLETE,0.89450,0.89342
3,52966806,A6_decision_tree_playground_series_s4e2.csv,2026-05-23 21:27:34.260000,DDS-8555 A6 decision tree model,SubmissionStatus.COMPLETE,0.87680,0.87030


,submission_file,exists_locally
0,A6_decision_tree_playground_series_s4e2.csv,True
1,A6_bagging_playground_series_s4e2.csv,True
2,A6_random_forest_playground_series_s4e2.csv,True
3,A6_gradient_boosting_playground_series_s4e2.csv,True


,evidence,value
0,Public GitHub repository,https://github.com/maldo81/dds-8555-predictive...
1,Notebook path in repository,Week 6/Assignment 6/MaldonadoJDDS8555-6.ipynb


## Interpretation

The tree family shows the bias-variance trade-off in a practical way.  A single decision tree is interpretable but unstable.  Bagging reduces variance by averaging many trees, random forests add feature randomness to reduce tree correlation, and boosting builds a sequence of trees that focus on difficult cases (Breiman, 1996, 2001; Friedman, 2001).  The validation table includes the train-validation gap so overfitting is visible rather than assumed.  The confusion matrix and feature-importance table give the best validation model a direct interpretation.  The Kaggle evidence shows that all four required submissions completed, with boosted trees producing the best public score among this group.  As in Assignment 5, these are competition obesity labels rather than clinical diagnoses (NCD Risk Factor Collaboration, 2016).

## References

Breiman, L. (1996).  Bagging predictors. *Machine Learning, 24*, 123-140. https://doi.org/10.1007/BF00058655

Breiman, L. (2001).  Random forests. *Machine Learning, 45*, 5-32. https://doi.org/10.1023/A:1010933404324

Friedman, J.  H. (2001).  Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189-1232. https://doi.org/10.1214/aos/1013203451

NCD Risk Factor Collaboration. (2016).  Trends in adult body-mass index in 200 countries from 1975 to 2014: A pooled analysis of 1698 population-based measurement studies with 19.2 million participants. *The Lancet, 387*(10026), 1377-1396. https://doi.org/10.1016/S0140-6736(16)30054-X